#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from joblib import dump, load

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

#### Loading the Prepared Forecasting Datasets

In [2]:
processed_dir = Path("../data/processed")
models_dir = Path("../models")
artifacts_dir = Path("../artifacts")

models_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

daily_df = pd.read_csv(
    processed_dir / "daily_reservations.csv",
    parse_dates=["arrival_datetime"]
)

train_df = pd.read_csv(
    processed_dir / "train_daily.csv",
    parse_dates=["arrival_datetime"]
)

test_df = pd.read_csv(
    processed_dir / "test_daily.csv",
    parse_dates=["arrival_datetime"]
)

train_features_df = pd.read_csv(
    processed_dir / "train_features.csv",
    parse_dates=["arrival_datetime"]
)

test_features_df = pd.read_csv(
    processed_dir / "test_features.csv",
    parse_dates=["arrival_datetime"]
)

target_column = "scheduled_reservations"
date_column = "arrival_datetime"
forecast_horizon = 7

feature_columns = [
    column
    for column in train_features_df.columns
    if column not in [date_column, target_column]
]

print("Forecasting Datasets Loaded Successfully")

print("\nOriginal Daily Series:")
print(f"Complete Dataset: {daily_df.shape}")
print(f"Training Dataset: {train_df.shape}")
print(f"Testing Dataset: {test_df.shape}")

print("\nFeature-Engineered Datasets:")
print(f"Training Dataset: {train_features_df.shape}")
print(f"Testing Dataset: {test_features_df.shape}")
print(f"Predictor Features: {len(feature_columns)}")

print("\nForecasting Configuration:")
print(f"Target Variable: {target_column}")
print(f"Forecast Horizon: {forecast_horizon} days")

print("\nArtifact Directories:")
print(f"Models Directory: {models_dir}")
print(f"Artifacts Directory: {artifacts_dir}")

Forecasting Datasets Loaded Successfully

Original Daily Series:
Complete Dataset: (549, 2)
Training Dataset: (439, 2)
Testing Dataset: (110, 2)

Feature-Engineered Datasets:
Training Dataset: (405, 26)
Testing Dataset: (110, 26)
Predictor Features: 24

Forecasting Configuration:
Target Variable: scheduled_reservations
Forecast Horizon: 7 days

Artifact Directories:
Models Directory: ..\models
Artifacts Directory: ..\artifacts


#### Key Observations
- **Successful Dataset Loading:** The original daily series contains 439 training and 110 testing observations, while the feature-engineered datasets contain 405 training and 110 testing observations.
- **Modeling Configuration Confirmed:** All 24 engineered predictors are available, the forecasting horizon is seven days, and the directories for saving models and artifacts have been created.

#### Validating Forecasting Dataset Integrity

In [3]:
validation_results = {
    "Original Training Dates Sorted": (
        train_df[date_column].is_monotonic_increasing
    ),
    "Original Testing Dates Sorted": (
        test_df[date_column].is_monotonic_increasing
    ),
    "Feature Training Dates Sorted": (
        train_features_df[date_column].is_monotonic_increasing
    ),
    "Feature Testing Dates Sorted": (
        test_features_df[date_column].is_monotonic_increasing
    ),
    "Chronological Separation": (
        train_df[date_column].max() < test_df[date_column].min()
    ),
    "Feature Columns Consistent": (
        train_features_df.columns.equals(test_features_df.columns)
    ),
    "No Missing Training Features": (
        train_features_df[feature_columns].notna().all().all()
    ),
    "No Missing Training Targets": (
        train_df[target_column].notna().all()
    ),
    "Training Targets Preserved": (
        train_features_df.set_index(date_column)[target_column].equals(
            train_df.set_index(date_column)[target_column].loc[
                train_features_df[date_column]
            ].set_axis(train_features_df[date_column])
        )
    ),
    "Original Daily Series Reconstructed": (
        pd.concat(
            [train_df, test_df],
            ignore_index=True
        ).equals(daily_df)
    )
}

print("Forecasting Dataset Integrity Validation:")
for check, result in validation_results.items():
    print(f"{check}: {result}")

print("\nOverall Validation:")
print(f"All Checks Passed: {all(validation_results.values())}")

Forecasting Dataset Integrity Validation:
Original Training Dates Sorted: True
Original Testing Dates Sorted: True
Feature Training Dates Sorted: True
Feature Testing Dates Sorted: True
Chronological Separation: True
Feature Columns Consistent: True
No Missing Training Features: True
No Missing Training Targets: True
Training Targets Preserved: True
Original Daily Series Reconstructed: True

Overall Validation:
All Checks Passed: True


#### Key Observations
- **Dataset Integrity Confirmed:** All original and feature-engineered datasets passed validation, with consistent feature structures, preserved target values, and correctly ordered dates.
- **Chronological Separation Maintained:** The training and testing periods remain independent, and the complete daily series can be reconstructed without losing observations.

#### Defining Expanding-Window Validation Folds

In [4]:
initial_train_size = 280
validation_step = forecast_horizon

validation_folds = []

for train_end in range(
    initial_train_size,
    len(train_df) - forecast_horizon + 1,
    validation_step
):
    validation_end = train_end + forecast_horizon

    validation_folds.append({
        "fold": len(validation_folds) + 1,
        "train_start": train_df[date_column].iloc[0],
        "train_end": train_df[date_column].iloc[train_end - 1],
        "validation_start": train_df[date_column].iloc[train_end],
        "validation_end": train_df[date_column].iloc[validation_end - 1],
        "train_size": train_end,
        "validation_size": forecast_horizon
    })

validation_folds_df = pd.DataFrame(validation_folds)

print("Expanding-Window Validation Configuration:")
print(f"Initial Training Size: {initial_train_size} days")
print(f"Forecast Horizon: {forecast_horizon} days")
print(f"Validation Step: {validation_step} days")
print(f"Total Validation Folds: {len(validation_folds_df)}")

print("\nValidation Fold Summary:")
print(validation_folds_df.to_string(index=False))

print("\nValidation Integrity:")
print(
    "All Validation Windows Have Seven Days:",
    validation_folds_df["validation_size"].eq(forecast_horizon).all()
)

print(
    "All Training Periods Precede Validation:",
    (
        validation_folds_df["train_end"]
        < validation_folds_df["validation_start"]
    ).all()
)

print(
    "Validation Remains Within Training Period:",
    (
        validation_folds_df["validation_end"]
        <= train_df[date_column].max()
    ).all()
)

Expanding-Window Validation Configuration:
Initial Training Size: 280 days
Forecast Horizon: 7 days
Validation Step: 7 days
Total Validation Folds: 22

Validation Fold Summary:
 fold train_start  train_end validation_start validation_end  train_size  validation_size
    1  2017-07-01 2018-04-06       2018-04-07     2018-04-13         280                7
    2  2017-07-01 2018-04-13       2018-04-14     2018-04-20         287                7
    3  2017-07-01 2018-04-20       2018-04-21     2018-04-27         294                7
    4  2017-07-01 2018-04-27       2018-04-28     2018-05-04         301                7
    5  2017-07-01 2018-05-04       2018-05-05     2018-05-11         308                7
    6  2017-07-01 2018-05-11       2018-05-12     2018-05-18         315                7
    7  2017-07-01 2018-05-18       2018-05-19     2018-05-25         322                7
    8  2017-07-01 2018-05-25       2018-05-26     2018-06-01         329                7
    9  2017-0

#### Key Observations
- **Expanding-Window Validation Established:** The framework contains 22 validation folds, beginning with 280 training observations and expanding by seven days after each forecast.
- **Forecasting Integrity Confirmed:** Every validation window contains exactly seven consecutive days, with training observations strictly preceding their corresponding validation periods.
- **Test Data Isolation Maintained:** All validation folds remain within the original training period, preserving the final 110-day testing period for independent evaluation.

#### Defining Forecasting Evaluation Metrics

In [5]:
def evaluate_forecast(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if y_true.shape != y_pred.shape:
        raise ValueError("Actual and predicted values must have matching shapes.")

    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError("Actual and predicted values must be finite.")

    y_pred = np.maximum(y_pred, 0)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    actual_total = np.abs(y_true).sum()

    wape = (
        np.abs(y_true - y_pred).sum() / actual_total * 100
        if actual_total > 0
        else np.nan
    )

    r2 = (
        r2_score(y_true, y_pred)
        if len(y_true) >= 2
        else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "R2": r2
    }


print("Forecasting Evaluation Function Defined Successfully")

print("\nValidation Example:")

actual = [50, 60, 70, 80, 90, 100, 110]
predicted = [55, 65, 75, 85, 95, 105, 115]

example_metrics = evaluate_forecast(actual, predicted)

for metric, value in example_metrics.items():
    print(f"{metric}: {value:.4f}")

Forecasting Evaluation Function Defined Successfully

Validation Example:
MAE: 5.0000
RMSE: 5.0000
WAPE: 6.2500
R2: 0.9375


#### Key Observations
- **Evaluation Framework Validated:** The reusable function successfully calculates MAE, RMSE, WAPE, and R², with an expected MAE and RMSE of 5.00 for the sample predictions.
- **Consistent Forecast Evaluation:** The function validates prediction shapes and finite values while enforcing nonnegative forecasts for reservation counts.

#### Developing Baseline Forecasting Models

In [6]:
def generate_baseline_forecasts(history, horizon=7):
    history = np.asarray(history, dtype=float)

    if len(history) < 7:
        raise ValueError("At least seven historical observations are required.")

    forecasts = {
        "Naive": np.repeat(history[-1], horizon),
        "Seasonal Naive": np.resize(history[-7:], horizon),
        "Historical Moving Average": np.repeat(
            history[-7:].mean(), horizon
        )
    }

    return forecasts


baseline_predictions = []
baseline_results = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    forecasts = generate_baseline_forecasts(
        history,
        horizon=forecast_horizon
    )

    for model_name, predicted in forecasts.items():
        metrics = evaluate_forecast(actual, predicted)

        baseline_results.append({
            "Model": model_name,
            "Fold": int(fold["fold"]),
            **metrics
        })

        for step, (actual_value, predicted_value) in enumerate(
            zip(actual, predicted),
            start=1
        ):
            baseline_predictions.append({
                "Model": model_name,
                "Fold": int(fold["fold"]),
                "Forecast_Step": step,
                "Actual": actual_value,
                "Predicted": max(float(predicted_value), 0.0)
            })

baseline_results_df = pd.DataFrame(baseline_results)
baseline_predictions_df = pd.DataFrame(baseline_predictions)

print("Baseline Forecasting Models Evaluated Successfully")

print("\nValidation Summary:")
print(f"Models Evaluated: {baseline_results_df['Model'].nunique()}")
print(f"Validation Folds: {baseline_results_df['Fold'].nunique()}")
print(f"Total Model-Fold Evaluations: {len(baseline_results_df)}")
print(f"Total Forecast Records: {len(baseline_predictions_df)}")

print("\nAverage Validation Metrics:")
baseline_summary_df = (
    baseline_results_df
    .groupby("Model")[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .sort_values("MAE")
    .round(4)
)

print(baseline_summary_df.to_string())

Baseline Forecasting Models Evaluated Successfully

Validation Summary:
Models Evaluated: 3
Validation Folds: 22
Total Model-Fold Evaluations: 66
Total Forecast Records: 462

Average Validation Metrics:
                               MAE     RMSE    WAPE      R2
Model                                                      
Historical Moving Average  25.6809  30.4130  27.867 -0.2786
Seasonal Naive             31.5714  38.5895  34.259 -1.6664
Naive                      37.2143  43.6073  40.322 -3.6521


#### Key Observations
- **Historical Moving Average Performs Best:** The seven-day moving average achieves the lowest MAE (25.68), RMSE (30.41), and WAPE (27.87%) among the three baseline models.
- **Simple Forecasting Methods Show Limitations:** Seasonal Naive (MAE 31.57) outperforms Naive (MAE 37.21), but all three models produce negative average fold-level R² values, indicating difficulty capturing short-term reservation fluctuations.

#### Developing Simple Exponential Smoothing Forecasts

In [7]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

ses_results = []
ses_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = SimpleExpSmoothing(
        history,
        initialization_method="estimated"
    )

    fitted_model = model.fit(
        optimized=True
    )

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    ses_results.append({
        "Model": "Simple Exponential Smoothing",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        ses_predictions.append({
            "Model": "Simple Exponential Smoothing",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

ses_results_df = pd.DataFrame(ses_results)
ses_predictions_df = pd.DataFrame(ses_predictions)

print("Simple Exponential Smoothing Evaluated Successfully")

print("\nValidation Summary:")
print(f"Validation Folds: {len(ses_results_df)}")
print(f"Total Forecast Records: {len(ses_predictions_df)}")

print("\nAverage Validation Metrics:")
print(
    ses_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_string()
)

print("\nComparison with Best Baseline:")
best_baseline_mae = baseline_summary_df["MAE"].min()
ses_mae = ses_results_df["MAE"].mean()

print(f"Historical Moving Average MAE: {best_baseline_mae:.4f}")
print(f"Simple Exponential Smoothing MAE: {ses_mae:.4f}")
print(f"MAE Improvement: {best_baseline_mae - ses_mae:.4f}")

Simple Exponential Smoothing Evaluated Successfully

Validation Summary:
Validation Folds: 22
Total Forecast Records: 154

Average Validation Metrics:
MAE     25.6880
RMSE    30.1320
WAPE    27.8631
R2      -0.2483

Comparison with Best Baseline:
Historical Moving Average MAE: 25.6809
Simple Exponential Smoothing MAE: 25.6880
MAE Improvement: -0.0071


#### Key Observations
- **Comparable Forecasting Accuracy:** Simple Exponential Smoothing achieves an MAE of 25.69 and RMSE of 30.13, performing almost identically to the Historical Moving Average baseline.
- **Limited Improvement Over Baselines:** Although SES slightly reduces RMSE, its MAE remains marginally higher than the best baseline, indicating no meaningful improvement in average absolute forecasting accuracy.

#### Consolidating Model Validation Results

In [9]:
model_results_df = pd.concat(
    [baseline_results_df, ses_results_df],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [baseline_predictions_df, ses_predictions_df],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Consolidated Model Results Initialized Successfully")

print("\nCurrent Model Comparison:")
print(model_comparison_df.round(4).to_string(index=False))

print("\nConsolidated Results Summary:")
print(f"Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Fold-Level Evaluations: {len(model_results_df)}")
print(f"Individual Forecasts: {len(model_predictions_df)}")

print("\nSaved Artifacts:")
print("model_validation_results.csv")
print("model_validation_predictions.csv")
print("model_comparison.csv")

Consolidated Model Results Initialized Successfully

Current Model Comparison:
                       Model     MAE    RMSE    WAPE      R2  Validation_Folds
   Historical Moving Average 25.6809 30.4130 27.8670 -0.2786                22
Simple Exponential Smoothing 25.6880 30.1320 27.8631 -0.2483                22
              Seasonal Naive 31.5714 38.5895 34.2590 -1.6664                22
                       Naive 37.2143 43.6073 40.3220 -3.6521                22

Consolidated Results Summary:
Models Evaluated: 4
Fold-Level Evaluations: 88
Individual Forecasts: 616

Saved Artifacts:
model_validation_results.csv
model_validation_predictions.csv
model_comparison.csv


#### Developing Holt's Linear Trend Forecasts

In [10]:
from statsmodels.tsa.holtwinters import Holt

holt_results = []
holt_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = Holt(
        history,
        exponential=False,
        damped_trend=True,
        initialization_method="estimated"
    )

    fitted_model = model.fit(
        optimized=True
    )

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    holt_results.append({
        "Model": "Holt Linear Trend",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        holt_predictions.append({
            "Model": "Holt Linear Trend",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

holt_results_df = pd.DataFrame(holt_results)
holt_predictions_df = pd.DataFrame(holt_predictions)

print("Holt's Linear Trend Evaluated Successfully")

print("\nValidation Summary:")
print(f"Validation Folds: {len(holt_results_df)}")
print(f"Total Forecast Records: {len(holt_predictions_df)}")

print("\nAverage Validation Metrics:")
print(
    holt_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_string()
)

print("\nComparison with Best Existing Model:")

best_existing_model = model_comparison_df.iloc[0]
holt_mae = holt_results_df["MAE"].mean()

print(f"Best Existing Model: {best_existing_model['Model']}")
print(f"Best Existing MAE: {best_existing_model['MAE']:.4f}")
print(f"Holt Linear Trend MAE: {holt_mae:.4f}")
print(f"MAE Improvement: {best_existing_model['MAE'] - holt_mae:.4f}")

Holt's Linear Trend Evaluated Successfully

Validation Summary:
Validation Folds: 22
Total Forecast Records: 154

Average Validation Metrics:
MAE     25.6826
RMSE    30.1377
WAPE    27.8564
R2      -0.2489

Comparison with Best Existing Model:
Best Existing Model: Historical Moving Average
Best Existing MAE: 25.6809
Holt Linear Trend MAE: 25.6826
MAE Improvement: -0.0017


In [11]:
model_results_df = pd.concat([model_results_df, holt_results_df], ignore_index=True)
model_predictions_df = pd.concat([model_predictions_df, holt_predictions_df], ignore_index=True)

In [12]:
display(model_comparison_df)

,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Historical Moving Average,25.680891,30.413028,27.866991,-0.278649,22
1,Simple Exponential Smoothing,25.688002,30.131954,27.863129,-0.248315,22
2,Seasonal Naive,31.571429,38.589536,34.258952,-1.666351,22
3,Naive,37.214286,43.607331,40.322012,-3.652148,22


#### Developing Holt-Winters Exponential Smoothing Forecasts

In [13]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

holt_winters_results = []
holt_winters_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = ExponentialSmoothing(
        history,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    holt_winters_results.append({
        "Model": "Holt-Winters",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        holt_winters_predictions.append({
            "Model": "Holt-Winters",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

holt_winters_results_df = pd.DataFrame(holt_winters_results)
holt_winters_predictions_df = pd.DataFrame(holt_winters_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Holt-Winters"
        ],
        holt_winters_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Holt-Winters"
        ],
        holt_winters_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Holt-Winters Exponential Smoothing Evaluated Successfully")

print("\nAverage Validation Metrics:")
display(
    holt_winters_results_df[
        ["MAE", "RMSE", "WAPE", "R2"]
    ].mean().round(4).to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

Holt-Winters Exponential Smoothing Evaluated Successfully

Average Validation Metrics:


,Value
MAE,26.0370
RMSE,30.7946
WAPE,28.4497
R2,-0.3557



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
1,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
2,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
3,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
4,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
5,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 6


#### Key Observations
- **Historical Moving Average Remains the Best Model:** It achieves the lowest MAE (25.68), narrowly outperforming Holt's Linear Trend (25.68) and Simple Exponential Smoothing (25.69).
- **Weekly Seasonality Provides Limited Improvement:** Holt-Winters achieves an MAE of 26.04, indicating that incorporating additive weekly seasonality and trend does not improve forecasting accuracy over the simpler approaches.

#### Developing ARIMA Forecasts

In [14]:
from statsmodels.tsa.arima.model import ARIMA

arima_results = []
arima_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = ARIMA(
        history,
        order=(1, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit()

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(steps=forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    arima_results.append({
        "Model": "ARIMA",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        arima_predictions.append({
            "Model": "ARIMA",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

arima_results_df = pd.DataFrame(arima_results)
arima_predictions_df = pd.DataFrame(arima_predictions)

model_results_df = pd.concat(
    [
        model_results_df[model_results_df["Model"] != "ARIMA"],
        arima_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[model_predictions_df["Model"] != "ARIMA"],
        arima_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("ARIMA Evaluated Successfully")

print("\nARIMA Average Validation Metrics:")
display(
    arima_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

ARIMA Evaluated Successfully

ARIMA Average Validation Metrics:


,Value
MAE,25.6696
RMSE,30.1731
WAPE,27.8351
R2,-0.2533



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
1,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
2,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
3,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
4,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
5,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
6,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 7


#### Key Observations
- **ARIMA Achieves the Lowest Forecasting Error:** ARIMA records the best MAE (25.67) and WAPE (27.84%) among the seven evaluated models, marginally outperforming the Historical Moving Average baseline.
- **Performance Improvements Remain Limited:** ARIMA's MAE improvement over the strongest baseline is only 0.01 reservations, while its RMSE (30.17) remains slightly higher than that of Simple Exponential Smoothing (30.13).

#### Developing Seasonal ARIMA Forecasts

In [15]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_results = []
sarima_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = SARIMAX(
        history,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        trend="n",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit(
        disp=False,
        maxiter=100
    )

    predicted = np.maximum(
        np.asarray(
            fitted_model.forecast(steps=forecast_horizon)
        ),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    sarima_results.append({
        "Model": "SARIMA",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        sarima_predictions.append({
            "Model": "SARIMA",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

sarima_results_df = pd.DataFrame(sarima_results)
sarima_predictions_df = pd.DataFrame(sarima_predictions)

model_results_df = pd.concat(
    [
        model_results_df[model_results_df["Model"] != "SARIMA"],
        sarima_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "SARIMA"
        ],
        sarima_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("SARIMA Evaluated Successfully")

print("\nSARIMA Average Validation Metrics:")
display(
    sarima_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

SARIMA Evaluated Successfully

SARIMA Average Validation Metrics:


,Value
MAE,25.3638
RMSE,30.1642
WAPE,27.6425
R2,-0.2890



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
6,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
7,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 8


#### Key Observations
- **SARIMA Achieves the Best Forecasting Accuracy:** SARIMA records the lowest MAE (25.36) and WAPE (27.64%), outperforming ARIMA (25.67) and the Historical Moving Average baseline (25.68).
- **Seasonal Modeling Provides a Modest Improvement:** Incorporating weekly seasonal autoregressive relationships reduces average forecasting error, although SARIMA's RMSE (30.16) and negative average R² indicate that substantial prediction errors remain.

#### Developing SARIMAX Forecasts with Calendar Features

In [16]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

calendar_df = train_df[[date_column, target_column]].copy()

calendar_df["month"] = calendar_df[date_column].dt.month
calendar_df["day_of_week"] = calendar_df[date_column].dt.dayofweek
calendar_df["is_weekend"] = (
    calendar_df["day_of_week"] >= 5
).astype(int)

calendar_exog = pd.get_dummies(
    calendar_df[["month", "day_of_week"]],
    columns=["month", "day_of_week"],
    drop_first=True,
    dtype=float
)

calendar_exog["is_weekend"] = calendar_df["is_weekend"].astype(float)

sarimax_results = []
sarimax_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])
    validation_end = train_end + forecast_horizon

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:validation_end
    ].to_numpy()

    exog_train = calendar_exog.iloc[:train_end]
    exog_validation = calendar_exog.iloc[
        train_end:validation_end
    ]

    model = SARIMAX(
        history,
        exog=exog_train,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        trend="n",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit(
        disp=False,
        maxiter=100
    )

    predicted = np.maximum(
        np.asarray(
            fitted_model.forecast(
                steps=forecast_horizon,
                exog=exog_validation
            )
        ),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    sarimax_results.append({
        "Model": "SARIMAX",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        sarimax_predictions.append({
            "Model": "SARIMAX",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

sarimax_results_df = pd.DataFrame(sarimax_results)
sarimax_predictions_df = pd.DataFrame(sarimax_predictions)

model_results_df = pd.concat(
    [
        model_results_df[model_results_df["Model"] != "SARIMAX"],
        sarimax_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "SARIMAX"
        ],
        sarimax_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("SARIMAX Evaluated Successfully")

print("\nSARIMAX Average Validation Metrics:")
display(
    sarimax_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

C:\Users\user\AppData\Local\Temp\ipykernel_204\276153991.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar_df["day_of_week"] = calendar_df[date_column].dt.dayofweek
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\Projects\ML and Data Science\StaySens

SARIMAX Evaluated Successfully

SARIMAX Average Validation Metrics:


d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,Value
MAE,37.4344
RMSE,43.5902
WAPE,41.1529
R2,-4.4552



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
6,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
7,Naive,37.2143,43.6073,40.3220,-3.6521,22
8,SARIMAX,37.4344,43.5902,41.1529,-4.4552,22



Total Models Evaluated: 9


#### Key Observations
- **SARIMA Remains the Best-Performing Model:** SARIMA achieves the lowest MAE (25.36) and WAPE (27.64%) among all nine evaluated models.
- **Calendar-Based SARIMAX Underperforms:** SARIMAX records an MAE of 37.43 and RMSE of 43.59, indicating that the selected calendar predictors and model configuration do not improve forecasting accuracy.